<a href="https://colab.research.google.com/github/harikarodda36-sys/Student-Performance-Prediction-and-Success-AI/blob/main/Another_copy_of_AICW_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd

In [6]:
df = pd.read_csv('/content/student_performance.csv')

In [7]:
df

,StudyHours,Attendance,Resources,Extracurricular,Motivation,Internet,Gender,Age,LearningStyle,OnlineCourses,Discussions,AssignmentCompletion,ExamScore,EduTech,StressLevel,FinalGrade
0,19,64,1,0,0,1,0,19,2,8,1,59,40,0,1,3
1,19,64,1,0,0,1,0,23,3,16,0,90,66,0,1,2
2,19,64,1,0,0,1,0,28,1,19,0,67,99,1,1,0
3,19,64,1,1,0,1,0,19,2,8,1,59,40,0,1,3
4,19,64,1,1,0,1,0,23,3,16,0,90,66,0,1,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13998,30,62,0,1,1,1,0,22,2,2,1,100,71,1,2,1
13999,30,62,0,1,1,1,0,23,3,12,1,72,55,1,1,2
14000,22,90,2,0,1,1,0,23,3,0,1,80,56,0,0,2
14001,22,90,2,0,1,1,0,29,2,16,0,50,62,1,2,2


In [8]:
# Remove leading/trailing spaces and convert to lowercase for consistency
df.columns = [col.strip().lower() for col in df.columns]

print("Cleaned Columns:", df.columns.tolist())

Cleaned Columns: ['studyhours', 'attendance', 'resources', 'extracurricular', 'motivation', 'internet', 'gender', 'age', 'learningstyle', 'onlinecourses', 'discussions', 'assignmentcompletion', 'examscore', 'edutech', 'stresslevel', 'finalgrade']


In [9]:
# Check for duplicate rows
duplicate_count = df.duplicated().sum()
print(f"Number of duplicate rows found: {duplicate_count}")

# Remove duplicates (keep only the first occurrence)
df = df.drop_duplicates()

Number of duplicate rows found: 1534


In [10]:
# Fill missing numeric values (like ExamScore or Attendance) with the median
numeric_cols = df.select_dtypes(include=['number']).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

# Fill missing categorical values (like Gender or LearningStyle) with the mode
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

/tmp/ipykernel_848/1536379090.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())


In [11]:
# Convert a column to numeric, turning errors (like '??' or letters) into NaN
df['examscore'] = pd.to_numeric(df['examscore'], errors='coerce')

# Re-fill any new NaNs created by the conversion
df['examscore'] = df['examscore'].fillna(df['examscore'].median())

/tmp/ipykernel_848/4028676437.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['examscore'] = pd.to_numeric(df['examscore'], errors='coerce')
/tmp/ipykernel_848/4028676437.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['examscore'] = df['examscore'].fillna(df['examscore'].median())


In [12]:
# Remove rows where attendance is mathematically impossible
df = df[(df['attendance'] >= 0) & (df['attendance'] <= 100)]

# Verify the final shape of the cleaned data
print(f"Final dataset shape: {df.shape}")

Final dataset shape: (12469, 16)


In [13]:
# Save the cleaned file without the index column
df.to_csv('cleaned_student_performance.csv', index=False)
print("✅ Cleaned data saved successfully!")

✅ Cleaned data saved successfully!


In [14]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
import pickle

# Load your dataset
df = pd.read_csv('student_performance.csv')
X = df.drop(columns=['FinalGrade'])
y = df['FinalGrade']

# Train the model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X, y)

# Save the model
with open('student_performance_model.pkl', 'wb') as f:
    pickle.dump(model, f)

print("✅ Step 1: Model is ready.")

✅ Step 1: Model is ready.


In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [16]:
y_pred = model.predict(X_test)

In [17]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

1.0

In [18]:
%%writefile app.py
import streamlit as st
import pickle
import pandas as pd

# Load model
model = pickle.load(open('student_performance_model.pkl', 'rb'))

st.set_page_config(page_title="Student Success AI", layout="wide")
st.title("🎓 Student Success Predictor")

# The Form with the Submit Button
with st.form("input_form"):
    st.subheader("Enter All 15 Student Metrics")
    c1, c2, c3 = st.columns(3)

    with c1:
        study = st.slider("Study Hours", 5, 45, 20)
        attendance = st.slider("Attendance %", 60, 100, 85)
        exam = st.number_input("Exam Score", 40, 100, 75)
        assignments = st.slider("Assignment Completion %", 50, 100, 80)
        online = st.number_input("Online Courses", 0, 20, 2)
    with c2:
        age = st.number_input("Age", 18, 30, 21)
        gender = st.selectbox("Gender (0:M, 1:F)", [0, 1])
        style = st.selectbox("Learning Style (0-3)", [0, 1, 2, 3])
        motivation = st.select_slider("Motivation (0-2)", options=[0, 1, 2])
        stress = st.select_slider("Stress (0-2)", options=[0, 1, 2])
    with c3:
        internet = st.radio("Internet Access", [1, 0])
        resources = st.selectbox("Resources (0-2)", [0, 1, 2])
        extra = st.radio("Extracurricular", [1, 0])
        edutech = st.radio("EduTech Tools", [1, 0])
        discussions = st.radio("Discussions", [1, 0])

    # THE SUBMIT BUTTON
    submitted = st.form_submit_button("Predict Grade")

if submitted:
    # This list matches your CSV order exactly
    cols = ['StudyHours', 'Attendance', 'Resources', 'Extracurricular', 'Motivation',
            'Internet', 'Gender', 'Age', 'LearningStyle', 'OnlineCourses',
            'Discussions', 'AssignmentCompletion', 'ExamScore', 'EduTech', 'StressLevel']

    input_df = pd.DataFrame([[study, attendance, resources, extra, motivation, internet, gender,
                             age, style, online, discussions, assignments, exam, edutech, stress]], columns=cols)

    prediction = model.predict(input_df)[0]
    grades = {0: "A (Excellent)", 1: "B (Good)", 2: "C (Average)", 3: "D/F (At Risk)"}

    st.markdown("---")
    st.success(f"### Predicted Result: {grades[prediction]}")
    if prediction == 0: st.balloons()

Writing app.py


In [19]:
# 1. Install Streamlit
!pip install -q streamlit

# 2. Train the model on your 15 features
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
import pickle

df = pd.read_csv('student_performance.csv')
X = df.drop(columns=['FinalGrade'])
y = df['FinalGrade']

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X, y)

with open('student_performance_model.pkl', 'wb') as f:
    pickle.dump(model, f)

print("✅ Setup Complete: Streamlit installed and Model trained.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 96.3 MB/s eta 0:00:00
✅ Setup Complete: Streamlit installed and Model trained.


In [20]:
%%writefile app.py
import streamlit as st
import pickle
import pandas as pd

# Load the model
model = pickle.load(open('student_performance_model.pkl', 'rb'))

st.set_page_config(page_title="Student AI", layout="centered")
st.title("🎓 Student Success Predictor")
st.write("Enter the details below. The prediction button is at the bottom of the page.")

# 15 Simple Inputs
st.subheader("📝 Student Details")
study = st.slider("Weekly Study Hours", 5, 45, 20)
attendance = st.slider("Attendance %", 60, 100, 85)
exam = st.number_input("Last Exam Score", 40, 100, 75)
assignments = st.slider("Assignment Completion %", 50, 100, 80)
online_courses = st.number_input("Online Courses", 0, 20, 2)
age = st.number_input("Age", 18, 30, 20)
gender = st.selectbox("Gender", [0, 1], format_func=lambda x: "Male" if x==0 else "Female")
style = st.selectbox("Learning Style", [0, 1, 2, 3])
motivation = st.select_slider("Motivation Level", options=[0, 1, 2])
stress = st.select_slider("Stress Level", options=[0, 1, 2])
internet = st.radio("Internet Access", [1, 0], horizontal=True)
resources = st.selectbox("Resources Availability", [0, 1, 2])
extra = st.radio("Extracurricular?", [1, 0], horizontal=True)
edutech = st.radio("Uses EduTech?", [1, 0], horizontal=True)
discussions = st.radio("Participates in Discussions?", [1, 0], horizontal=True)

st.markdown("---")

# THE BIG VISIBLE BUTTON
# use_container_width makes it span the whole page
if st.button("🚀 CLICK HERE TO PREDICT GRADE", use_container_width=True):
    # column names must match your CSV exactly
    cols = ['StudyHours', 'Attendance', 'Resources', 'Extracurricular', 'Motivation',
            'Internet', 'Gender', 'Age', 'LearningStyle', 'OnlineCourses',
            'Discussions', 'AssignmentCompletion', 'ExamScore', 'EduTech', 'StressLevel']

    data = [[study, attendance, resources, extra, motivation, internet, gender,
             age, style, online_courses, discussions, assignments, exam, edutech, stress]]

    input_df = pd.DataFrame(data, columns=cols)
    prediction = model.predict(input_df)[0]

    grades = {0: "Excellent (A)", 1: "Good (B)", 2: "Average (C)", 3: "At Risk (D/F)"}
    st.success(f"### Predicted Result: {grades[prediction]}")
    if prediction == 0: st.balloons()

Overwriting app.py


In [ ]:
# 1. Get the Password for the Tunnel
import urllib
ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
print("------------------------------------------")
print("YOUR TUNNEL PASSWORD (IP):", ip)
print("------------------------------------------")

# 2. Run the app
!python -m streamlit run app.py & npx localtunnel --port 8501

------------------------------------------
YOUR TUNNEL PASSWORD (IP): 35.185.90.136
------------------------------------------
⠙⠹

⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) 
  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.185.90.136:8501

